# EXP-007 — Tối ưu siêu tham số KNN trên WESAD

**Người thực hiện:** Điền tên của bạn  
**Trạng thái:** Chưa chạy bộ dữ liệu thực tế (cần chạy R01 và R02)  
**Nhóm:** 02 — Hyperparameter Tuning

## 1. Mục tiêu và giả thuyết

Đánh giá liệu tối ưu KNN bằng **nested cross-validation theo người** có cải thiện khả năng phát hiện stress so với **EXP-003 KNN mặc định** không. Không kết luận trước khi có kết quả. Đo Macro-F1 và Balanced Accuracy trên từng subject, hai bài toán nhị phân và ba lớp.

## 2. Thiết kế thí nghiệm

- Bộ G2 được nhóm nghiệm thu: **68 đặc trưng sinh lý**, **15 người** (S2–S11, S13–S17).
- **Outer LOSO:** mỗi lần giữ 1 người test; 14 người còn lại huấn luyện.
- **Inner GroupKFold(3):** chọn siêu tham số chỉ trong 14 người train, không nhìn người test outer.
- Mỗi inner fold áp dụng SimpleImputer(median), StandardScaler và KNN **chỉ fit trên train**.
- Bốn ứng viên khóa trước: `(3, euclidean, uniform)`, `(5, manhattan, distance)`, `(9, euclidean, distance)`, `(5, euclidean, uniform)`.
- Điểm lựa chọn siêu tham số: **Macro-F1** (không dùng outer test để chọn tham số).
- Đối chứng: EXP-003 (KNN mặc định, `k=5`, Euclidean, uniform), cùng dữ liệu và outer folds.

**Lưu ý:** Các cửa sổ WESAD chồng lấn, không được tách train/test ngẫu nhiên theo cửa sổ.

## 3. Khởi tạo môi trường và đọc cấu hình

In [ ]:
from pathlib import Path
import sys, json, hashlib, subprocess

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'scripts/run_experiment.py').is_file() and (p / 'src/wesad/models.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không thấy thư mục gốc wesad-stress-detection. Mở notebook trong dự án đã giải nén.')
CONFIG_PATH = ROOT / 'experiments/02_tuning/EXP-007_KNN/config.yaml'
try:
    import yaml, pandas as pd, numpy as np, matplotlib.pyplot as plt
    import sklearn, imblearn
except ImportError as exc:
    raise ImportError(f'{exc}. Cài dependencies bằng Terminal tại ROOT: python -m pip install -r requirements.lock.txt; python -m pip install matplotlib ipykernel') from exc
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
DATA = ROOT / CONFIG['features']
SCHEMA = ROOT / CONFIG['schema']
MAIN_DIR = ROOT / CONFIG['output_dir']
QUICK_DIR = MAIN_DIR.parent / 'r03_quick_S2'
EXPORT_DIR = CONFIG_PATH.parent
print('ROOT:', ROOT)
print('Interpreter:', sys.executable)
print('Data:', DATA)
print('Output main:', MAIN_DIR)

## 4. Kiểm tra dữ liệu và checksum trước khi huấn luyện

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if not DATA.is_file() or not SCHEMA.is_file():
    raise FileNotFoundError('Thiếu bảng G2 physiological68 hoặc feature_schema.json. Xem docs/DU_LIEU_VA_TRUY_CAP.md')
assert sha256(DATA) == CONFIG['feature_sha256'], 'Checksum dữ liệu G2 không đúng bản nhóm đã chốt'
sys.path.insert(0, str(ROOT / 'src'))
from wesad.data import load_feature_table
frame, feature_columns = load_feature_table(DATA, SCHEMA)
assert len(feature_columns) == 68
assert set(frame.subject_id) == set(CONFIG['expected_subjects'])
assert not frame.duplicated(['subject_id','window_id']).any()
assert frame.label_original.isin([1,2,3]).all()
print('Số cửa sổ:', len(frame), 'Số người:', frame.subject_id.nunique(), 'Số features:', len(feature_columns))
display(pd.DataFrame({'label': frame.label_original.value_counts().sort_index().index,
                      'count': frame.label_original.value_counts().sort_index().values}))

## 5. Kiểm tra cấu hình mô hình và grid

In [ ]:
from wesad.models import build_pipeline, search_grid
from sklearn.model_selection import GroupKFold
pipe = build_pipeline('knn', CONFIG)
assert CONFIG['nested'] is True and CONFIG['balance'] is False
assert CONFIG['inner_splits'] == 3 and len(search_grid('knn', CONFIG)) == 4
print(pipe)
display(pd.DataFrame([{
    'n_neighbors': v['model__n_neighbors'][0],
    'metric': v['model__metric'][0],
    'weights': v['model__weights'][0]
} for v in CONFIG['search_spaces']['knn']]))

## 6. R01 — Chạy thử một fold trên S2

Bật `RUN_QUICK=True` khi đã chuẩn bị dữ liệu. Kết quả thử không được gọi là kết quả chính thức của 15 người.

In [ ]:
RUN_QUICK = False

def execute(args):
    cmd = [sys.executable, 'scripts/run_experiment.py', '--config', str(CONFIG_PATH), *args]
    print('Lệnh:', ' '.join(map(str,cmd)))
    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, encoding='utf-8', errors='replace')
    print('Mã thoát:', proc.returncode)
    if proc.stdout: print('STDOUT:\n' + proc.stdout[-12000:])
    if proc.stderr: print('STDERR:\n' + proc.stderr[-18000:])
    if proc.returncode:
        raise RuntimeError('Runner thất bại. Xem STDERR phía trên để xác định nguyên nhân; không xóa output khi chưa biết lỗi.')
    return proc

if RUN_QUICK:
    execute(['--output', str(QUICK_DIR), '--outer-subjects', 'S2', '--resume'])
else:
    print('Chưa chạy R01. Đặt RUN_QUICK=True nếu muốn chạy thử S2.')

## 7. R02 — Chạy nested CV chính thức đủ 15 người

Cần **15 outer folds × 2 tasks = 30 fold**. Mỗi outer fold thực hiện grid search 4 ứng viên × 3 inner folds. Lệnh sẽ dùng lại các fold đã kiểm tra nếu `--resume`. Không thay đổi cấu hình sau khi đã chạy trong cùng thư mục output.

In [ ]:
RUN_MAIN = False
if RUN_MAIN:
    execute(['--resume'])
else:
    print('Chưa chạy R02. Đặt RUN_MAIN=True sau khi R01 chạy ổn.')

## 8. Đọc kết quả đã kiểm chứng

In [ ]:
SUMMARY_PATH=MAIN_DIR/'summary.json'
READY=False
if SUMMARY_PATH.is_file():
    summary=json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
    if summary.get('status') == 'complete':
        scores=pd.read_csv(MAIN_DIR/'metrics_by_subject.csv',dtype={'subject_id':str})
        preds=pd.read_csv(MAIN_DIR/'predictions.csv',dtype={'subject_id':str,'window_id':str})
        READY=(len(scores)==30 and set(scores.subject_id)==set(CONFIG['expected_subjects'])
               and set(scores.task)=={'binary','multiclass'} and set(scores.model)=={'knn'})
        if READY:
            display(pd.DataFrame(summary['results'])[['task','model','macro_f1_mean','macro_f1_std',
                                                       'balanced_accuracy_mean','balanced_accuracy_std']])
if not READY:
    print('Chưa có kết quả R02 đầy đủ và hợp lệ. Không tự tạo các chỉ số giả.')

## 9. Xuất CSV, ảnh PNG và phân tích tham số tối ưu

In [ ]:
if READY:
    report_cmd=[sys.executable,'scripts/exp007_report.py','--run-dir',str(MAIN_DIR),
                '--export-dir',str(EXPORT_DIR),
                '--baseline-dir',str(ROOT/'outputs/notebook_research/EXP-003/r02_main')]
    proc=subprocess.run(report_cmd,cwd=ROOT,capture_output=True,text=True,encoding='utf-8',errors='replace')
    print(proc.stdout)
    if proc.returncode:
        print(proc.stderr)
        raise RuntimeError('Không xuất được báo cáo EXP-007. Xem thông báo lỗi trên.')
    display(pd.read_csv(EXPORT_DIR/'summary.csv'))
    display(pd.read_csv(EXPORT_DIR/'best_params_by_subject.csv').head())
else:
    print('Chỉ xuất file sau khi R02 chạy đủ 30 fold.')

## 10. So sánh ghép cặp với EXP-003 (nếu có dữ liệu đối chứng tương thích)

In [ ]:
comparison_file=EXPORT_DIR/'comparison_exp003.csv'
if comparison_file.exists() and READY:
    comparison=pd.read_csv(comparison_file)
    display(comparison.groupby('task')[["delta_macro_f1","delta_balanced_accuracy"]].agg(['mean','std']))
    print('Delta = EXP-007 tuned trừ EXP-003 default; số dương chỉ xu hướng cải thiện trong bộ WESAD này.')
else:
    print('Chưa có so sánh EXP-003 cùng dữ liệu. Có thể chạy EXP-003 rồi cập nhật đường dẫn baseline-dir ở cell 9.')

## 11. Nhận xét và kết luận (bổ sung sau khi chạy thật)

**Mẫu nhận xét để điền theo CSV:**

- Macro-F1 trung bình ± độ lệch chuẩn của KNN tuned trên 15 người đối với bài toán nhị phân: `...`.
- Macro-F1 trung bình ± độ lệch chuẩn đối với ba lớp: `...`.
- Các cấu hình thường được chọn nhất trong `best_params_by_subject.csv`: `...`.
- Chênh lệch Macro-F1 ghép cặp giữa EXP-007 và EXP-003: `...` (cần cùng dữ liệu và protocol).
- Những người có điểm thấp, biến động hiệu suất, vấn đề mất cân bằng nhãn: `...`.

**Kết luận chỉ viết sau khi có bằng chứng**: Không khẳng định tuning cải thiện nếu Delta âm/không đáng kể; không coi điểm trên một người hay inner CV là hiệu quả tổng quát. Thí nghiệm không thay đổi đặc trưng đầu vào hoặc áp dụng oversampling.
